# Stable Diffusion 1.5 + LoRA Fine-Tuning on Pokemon

Fine-tunes Stable Diffusion 1.5 with a low-rank (LoRA) adapter injected into the UNet attention layers, trained on the `svjack/pokemon-blip-captions-en-zh` dataset, to generate Pokemon-style creature artwork. The base model is kept fully frozen; only the LoRA weights are trained.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 2. Install dependencies

In [ ]:
!pip install -q -U diffusers transformers accelerate peft safetensors datasets
!pip uninstall -y torchao
print("done")

## 3. Verify versions and CUDA

In [ ]:
import importlib, torch

def _v(pkg):
    try: return importlib.import_module(pkg).__version__
    except Exception: return "NOT INSTALLED"

print(f"torch        {torch.__version__}")
print(f"diffusers    {_v('diffusers')}")
print(f"transformers {_v('transformers')}")
print(f"peft         {_v('peft')}")
print(f"accelerate   {_v('accelerate')}")
print(f"torchao      {_v('torchao')}   (expected: NOT INSTALLED)")
assert torch.cuda.is_available(), "No CUDA GPU - set Runtime -> T4 GPU"
print("\nCUDA OK ->", torch.cuda.get_device_name(0))

## 4. Load base SD 1.5 and generate a baseline image

Generating one image with the untouched base model gives a genuine before/after comparison once the LoRA adapter is trained.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"  # runwayml repo was removed; this is the mirror

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,        # half precision -> fits a T4 comfortably
    safety_checker=None,              # avoids the NSFW filter blanking legit images during training
    requires_safety_checker=False,    # suppresses the warning + skips loading checker weights
)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()       # trims VRAM further

prompt = "a photo of a corgi wearing a tiny wizard hat, studio lighting, sharp focus"
generator = torch.Generator("cuda").manual_seed(42)  # fixed seed = reproducible sampler

baseline = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=generator).images[0]
baseline.save("baseline_before_lora.png")
baseline  # displays inline

## 5. (Optional) Hugging Face login

Only needed for gated models or to raise Hub rate limits. A read-scope token from https://huggingface.co/settings/tokens is enough.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 6. Freeze the base model, inject a fresh LoRA adapter, load the dataset

In [ ]:
import torch, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset
from diffusers import DDPMScheduler, UNet2DConditionModel
from peft import LoraConfig

device = "cuda"
weight_dtype = torch.float16
RANK = 8  # sweep this later: 4 / 8 / 16 - change and re-run this cell onward, no restart needed

# The dataset to train on. Must be OPEN (no login / no access request).
DATASET_ID = "svjack/pokemon-blip-captions-en-zh"

# Reload a CLEAN UNet every run -> avoids "adapter 'default' already exists" on re-runs.
pipe.unet = UNet2DConditionModel.from_pretrained(
    MODEL_ID, subfolder="unet", torch_dtype=torch.float16
).to("cuda")
pipe.enable_attention_slicing()

unet, vae, text_encoder, tokenizer = pipe.unet, pipe.vae, pipe.text_encoder, pipe.tokenizer
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

# Freeze everything - we train ONLY the LoRA layers
vae.requires_grad_(False); text_encoder.requires_grad_(False); unet.requires_grad_(False)

# Inject a fresh, randomly-initialized LoRA adapter into the UNet attention blocks.
# init_lora_weights="gaussian": A ~ Gaussian, B = 0 -> adapter is a no-op at step 0.
lora_config = LoraConfig(
    r=RANK,
    lora_alpha=RANK,               # alpha=r -> scaling 1.0, so rank is the only variable in a sweep
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
)
unet.add_adapter(lora_config)

# Keep trainable LoRA params in fp32 for stable optimization (frozen base stays fp16)
for p in unet.parameters():
    if p.requires_grad:
        p.data = p.data.float()

unet.enable_gradient_checkpointing()
unet.train()

# ---- Load the dataset ----
dataset = load_dataset(DATASET_ID, split="train")
cols = dataset.column_names
img_col = next(c for c in ["image", "img"] if c in cols)
txt_col = next(c for c in ["text", "caption", "en_text", "en"] if c in cols)
dataset = dataset.rename_columns({img_col: "image", txt_col: "text"})
print(f"LOADED {DATASET_ID} | {len(dataset)} rows | image='{img_col}' text='{txt_col}'")

res = 512
tf = transforms.Compose([
    transforms.Resize(res, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(res),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

def preprocess(ex):
    ex["pixel_values"] = [tf(img.convert("RGB")) for img in ex["image"]]
    tok = tokenizer(ex["text"], max_length=tokenizer.model_max_length,
                     padding="max_length", truncation=True, return_tensors="pt")
    ex["input_ids"] = list(tok.input_ids)
    return ex

dataset = dataset.with_transform(preprocess)

def collate(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]).contiguous().float(),
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
    }

loader = DataLoader(dataset, shuffle=True, batch_size=1, collate_fn=collate)
n_trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f"{n_trainable:,} trainable LoRA params (rank={RANK}) | dataset: {DATASET_ID}")

## 7. Training loop

In [ ]:
lora_params = [p for p in unet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(lora_params, lr=1e-4)
scaler = torch.amp.GradScaler("cuda")

max_train_steps = 100
losses = []
step = 0
while step < max_train_steps:
    for batch in loader:
        pixel_values = batch["pixel_values"].to(device, dtype=weight_dtype)
        input_ids = batch["input_ids"].to(device)

        # Frozen encoders -> no grad. VAE stays OUT of autocast: SD's VAE is numerically
        # twitchy in fp16 and can emit NaNs; no need to risk it since it's frozen.
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample() * vae.config.scaling_factor
            enc_hidden = text_encoder(input_ids)[0]

        noise = torch.randn_like(latents)
        t = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                           (latents.shape[0],), device=device).long()
        noisy = noise_scheduler.add_noise(latents, noise, t)

        with torch.autocast("cuda", dtype=torch.float16):
            pred = unet(noisy, t, enc_hidden).sample
            target = noise  # SD 1.5 is epsilon-prediction
            loss = F.mse_loss(pred.float(), target.float())

        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        optimizer.zero_grad()

        losses.append(loss.item())
        step += 1
        if step % 10 == 0:
            print(f"step {step:3d} | loss {loss.item():.4f}")
        if step >= max_train_steps:
            break

print(f"\ndone - first loss {losses[0]:.4f} -> last loss {losses[-1]:.4f}")

## 8. (Optional) Plot the loss curve

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(losses)
plt.xlabel("step"); plt.ylabel("MSE loss"); plt.title(f"LoRA training loss (rank={RANK})")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"loss_curve_r{RANK}.png", dpi=120)
plt.show()

## 9. Generate images with the trained LoRA adapter

In [ ]:
import random
from PIL import Image

PROMPT = "a monster pokemon creature, sci-fi style"  # <-- change this to whatever you want
NUM_IMAGES = 4       # how many images to make
STEPS = 30
GUIDANCE = 7.5
SEED = None           # None = different each run; set an int (e.g. 42) for reproducible output

# If you RESTARTED the runtime and lost the in-memory adapter, uncomment to reload from disk:
# pipe.load_lora_weights(f"lora-r{RANK}")

pipe.unet.eval()

def image_grid(imgs, cols=2):
    rows = (len(imgs) + cols - 1) // cols
    imgs = imgs + [Image.new("RGB", imgs[0].size, "white")] * (rows * cols - len(imgs))
    w, h = imgs[0].size
    grid = Image.new("RGB", (cols * w, rows * h))
    for i, im in enumerate(imgs):
        grid.paste(im, (i % cols * w, i // cols * h))
    return grid

images = []
for i in range(NUM_IMAGES):
    seed = (SEED + i) if SEED is not None else random.randint(0, 2**32 - 1)
    g = torch.Generator("cuda").manual_seed(int(seed))
    img = pipe(PROMPT, num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=g).images[0]
    img.save(f"gen_{i}_seed{seed}.png")
    images.append(img)
    print(f"image {i}  seed {seed}")

grid = image_grid(images, cols=min(NUM_IMAGES, 2))
grid.save("generated_grid.png")
print(f'\nprompt: "{PROMPT}"  ->  saved generated_grid.png')
grid  # displays inline